# From-scratch adaptive-depth math transformer

This notebook trains a new model from random initialization on the local Hendrycks MATH parquet files. It does **not** load or modify Qwen/pretrained weights. Its decoder follows Qwen2/Qwen2.5 architecture patterns: pre-norm RMSNorm, RoPE (theta=1,000,000), grouped-query self-attention with Q/K/V biases, SwiGLU, separate token embeddings/output head, and causal next-token training. Attention uses PyTorch SDPA, which selects Flash Attention when supported by the GPU, dtype, and mask. Intermediate decoder exits are trained with auxiliary answer-token loss and can be selected adaptively at inference.

Run cells from top to bottom. Both the tokenizer and model are trained from scratch using only the training split; no pretrained weights or checkpoints are loaded.

In [1]:
from pathlib import Path
import importlib
import torch
from torch.utils.data import DataLoader
from tokenizers import Tokenizer
import scratch_qwen_adaptive
importlib.reload(scratch_qwen_adaptive)  # ensure notebook uses the current model/training implementation
from scratch_qwen_adaptive import (
    AdaptiveMathTransformer, MathSolutionDataset, ModelConfig,
    collate_math, evaluate, flash_attention_status, generate_solution,
    load_math_parquets, make_loaders, seed_everything, train_model, train_tokenizer,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print(flash_attention_status(device))

Device: cuda
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU. Attention calls use PyTorch SDPA, which dispatches to Flash Attention when the device, dtype, and mask are supported.


In [2]:
# Load the local subject parquet files. Keep the provided test split separate.
train_questions, train_answers, test_frame = load_math_parquets('hendrycks_math')
print('Training examples:', len(train_questions))
print('Test examples:', 0 if test_frame is None else len(test_frame))
print('Question sample:', train_questions.iloc[0][:250])
print('Solution sample:', train_answers.iloc[0][:250])

# Hold out 5% of the training examples for validation.
perm = torch.randperm(len(train_questions), generator=torch.Generator().manual_seed(42)).tolist()
n_valid = max(1, int(0.05 * len(perm)))
valid_idx, fit_idx = perm[:n_valid], perm[n_valid:]
fit_questions, fit_answers = train_questions.iloc[fit_idx].tolist(), train_answers.iloc[fit_idx].tolist()
valid_questions, valid_answers = train_questions.iloc[valid_idx].tolist(), train_answers.iloc[valid_idx].tolist()
print(f'Fit: {len(fit_questions):,} | validation: {len(valid_questions):,}')

Training examples: 7500
Test examples: 5000
Question sample: Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lif
Solution sample: For the piecewise function to be continuous, the cases must "meet" at $2$ and $-2$. For example, $ax+3$ and $x-5$ must be equal when $x=2$. This implies $a(2)+3=2-5$, which we solve to get $2a=-6 \Rightarrow a=-3$. Similarly, $x-5$ and $2x-b$ must be
Fit: 7,125 | validation: 375


In [3]:
# Train a byte-level BPE from scratch on BOTH math questions and solutions.
VOCAB_SIZE = 8000
BLOCK_SIZE = 512
BATCH_SIZE = 8
tokenizer_path = Path('scratch_math_tokenizer.json')
tokenizer = train_tokenizer(fit_questions, fit_answers, vocab_size=VOCAB_SIZE)
tokenizer.save(str(tokenizer_path))

train_loader, valid_loader = make_loaders(
    fit_questions, fit_answers, valid_questions, valid_answers,
    tokenizer, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE, num_workers=0
)
print('Vocabulary:', tokenizer.get_vocab_size())
print('Train batches:', len(train_loader), '| validation batches:', len(valid_loader))
# Confirm answer tokens are supervised and prompt/padding positions are ignored.
example_ids, example_labels, example_mask = next(iter(train_loader))
print('Batch shapes:', example_ids.shape, example_labels.shape, example_mask.shape)
print('Supervised answer tokens:', (example_labels[:, 1:] != -100).sum().item())

Vocabulary: 8000
Train batches: 891 | validation batches: 47
Batch shapes: torch.Size([8, 385]) torch.Size([8, 385]) torch.Size([8, 385])
Supervised answer tokens: 1132


In [4]:
# Compact scratch model; increase width/depth if GPU memory and dataset size permit.
config = ModelConfig(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=256,
    num_heads=8,
    num_kv_heads=2,  # grouped-query attention
    num_layers=6,
    max_seq_len=BLOCK_SIZE,
    exit_layers=(2, 4),
)
model = AdaptiveMathTransformer(config)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 8,328,704


## Train from random initialization

This is teacher-forced next-token training from **random weights**, not fine-tuning: no pretrained checkpoint is loaded. The input is `[BOS] [QUESTION] question [ANSWER] solution [EOS]`; loss is applied to the solution tokens and EOS, not the question or padding. Intermediate decoder exits receive auxiliary answer-token losses, so their predictions are trained rather than being uncalibrated early heads. Validation loss is token-weighted and early stopping keeps the best scratch-trained checkpoint.

In [5]:
best_val_loss = train_model(
    model, train_loader, valid_loader, device,
    epochs=30, learning_rate=3e-4, warmup_steps=200, patience=5,
    checkpoint='math_adaptive_scratch_best.pt'
)
print('Best validation token loss:', best_val_loss)

c:\Users\HP\.vscode\python\ml\llms\adaptive_depth_slm_for_maths\slm-for-math-using-adaptive-depth\scratch_qwen_adaptive.py:344: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)


epoch 001 | train/token 5.4130 | val/token 3.0988 | lr 3.00e-04
epoch 002 | train/token 3.5540 | val/token 2.7103 | lr 2.98e-04
epoch 003 | train/token 3.1453 | val/token 2.5142 | lr 2.94e-04
epoch 004 | train/token 2.8741 | val/token 2.3791 | lr 2.89e-04
epoch 005 | train/token 2.6673 | val/token 2.3063 | lr 2.83e-04
epoch 006 | train/token 2.4908 | val/token 2.2473 | lr 2.76e-04
epoch 007 | train/token 2.3368 | val/token 2.2289 | lr 2.67e-04
epoch 008 | train/token 2.1962 | val/token 2.2179 | lr 2.57e-04
epoch 009 | train/token 2.0622 | val/token 2.2377 | lr 2.46e-04
epoch 010 | train/token 1.9321 | val/token 2.2562 | lr 2.34e-04
epoch 011 | train/token 1.8065 | val/token 2.3143 | lr 2.22e-04
epoch 012 | train/token 1.6880 | val/token 2.3598 | lr 2.09e-04
epoch 013 | train/token 1.5708 | val/token 2.4151 | lr 1.95e-04
Early stopping: validation loss did not improve for 5 epochs.
Best validation loss: 2.2178508946673587


In [6]:
# Load the best validation checkpoint (still entirely trained from scratch).
checkpoint = torch.load('math_adaptive_scratch_best.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
model.to(device).eval()
print('Best epoch:', checkpoint['epoch'], '| validation token loss:', checkpoint['best_val_loss'])

# Optional held-out answer-token loss; the test set is not used for training.
if test_frame is not None and len(test_frame):
    test_questions = test_frame.iloc[:, 0].fillna('').astype(str).tolist()
    test_answers = test_frame.iloc[:, -1].fillna('').astype(str).tolist()
    test_ds = MathSolutionDataset(test_questions, test_answers, tokenizer, block_size=BLOCK_SIZE)
    pad_id = tokenizer.token_to_id('[PAD]')
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                             collate_fn=lambda rows: collate_math(rows, pad_id))
    print('Test answer-token loss:', evaluate(model, test_loader, device))

Best epoch: 8 | validation loss: 2.2178508946673587
Test answer-token loss: 2.0914727735033343


In [7]:
question = 'Solve: If 2x + 3 = 11, find x.'
# Compare full-depth and adaptive decoding on the same example.
answer = generate_solution(model, tokenizer, question, device,
                          max_new_tokens=128, use_adaptive=False)
adaptive_answer, layers_used = generate_solution(model, tokenizer, question, device,
                                   max_new_tokens=128, use_adaptive=True,
                                   confidence_threshold=0.95, adaptive_min_depth=2,
                                   return_depths=True)
print('Full depth:', answer)
print('Adaptive depth:', adaptive_answer)
print('Layers used per generated token:', layers_used)

We can write $x + \frac{11}{11} = 11(11) + \dotsb$.  We can solve for $x$ and $x$, so we have that $x = 11$, $11 = 11$.  Therefore, $x+3 = 11+11=\boxed{11}$.
